In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

from piano_tiles_ai.audio import load_audio
from piano_tiles_ai.separation import separate_piano
from piano_tiles_ai.transcription import transcribe_piano
from piano_tiles_ai.notes import load_notes

piano_path = separate_piano("../data/LetItBe.mp3", cache_dir="../outputs/separated")
midi_path = transcribe_piano(piano_path, cache_dir="../outputs/transcribed")
notes = load_notes(midi_path)

print(len(notes), "notes")
print(notes[:5])

In [ ]:
OFFSET, DURATION = 60.0, 20.0
hop = 512
fmin = librosa.note_to_hz("A0")

y, sr = load_audio(piano_path, offset=OFFSET, duration=DURATION)
C_db = librosa.amplitude_to_db(
    np.abs(librosa.cqt(y, sr=sr, hop_length=hop, fmin=fmin, n_bins=88)), ref=np.max
)

window = [n for n in notes if n.end > OFFSET and n.start < OFFSET + DURATION]

fig, ax = plt.subplots(figsize=(16, 7))
librosa.display.specshow(
    C_db, sr=sr, hop_length=hop, x_axis="time", y_axis="cqt_note", fmin=fmin, ax=ax
)
ax.hlines(
    y=[librosa.midi_to_hz(n.pitch) for n in window],
    xmin=[n.start - OFFSET for n in window],
    xmax=[n.end - OFFSET for n in window],
    colors="cyan",
    linewidth=2,
)
ax.set_title(f"CQT of the piano stem with transcribed notes ({OFFSET:.0f}–{OFFSET + DURATION:.0f} s)")
plt.show()

In [ ]:
pitches = [n.pitch for n in notes]

fig, ax = plt.subplots(figsize=(14, 4))
ax.hist(pitches, bins=range(21, 110))
ax.set_xlabel("MIDI pitch (60 = C4, middle C)")
ax.set_ylabel("Number of notes")
ax.set_title("How often each key is played")
plt.show()